In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

### TF-IDF Matching

In [2]:
df1 = pd.read_csv("../data/committees/corporate_pacs.csv")
df2 = pd.read_csv("../data/stocks/us_equities.csv", index_col=0).reset_index(drop=True)

In [128]:
def preprocess(text, business = True, politics = True, symbol = True, period = False):
    text = text.lower().strip()
    text = re.sub(r'\([^)]*\)', ' ', text)
    if symbol:
        text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    if period:
        text = re.sub(r'\.', ' ', text)
    if business:
        text = re.sub(r'\b(inc|corp|corporation|ltd|llc|group|co|plc|limited|company|incorporated|the|and|holdings)\b', ' ', text)
    if politics:
        text = re.sub(r'\b(federal|political|committee|action|election|politi|politic|pac)\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [4]:
df1["company_name"] = df1["CONNECTED_ORG_NM"].apply(preprocess)
df2["company_name"] = df2["DTSubjectName"].apply(preprocess)

In [5]:
vectorizer = TfidfVectorizer().fit(pd.concat([df1["company_name"], df2["company_name"]]))
tfidf_matrix_1 = vectorizer.transform(df1["company_name"])
tfidf_matrix_2 = vectorizer.transform(df2["company_name"])

cosine_sim = cosine_similarity(tfidf_matrix_1, tfidf_matrix_2)

In [ ]:
matches_tfidf = []
unmatched = []
for i, row in tqdm(enumerate(cosine_sim)):
    best_match_idx = row.argmax()
    best_score = row[best_match_idx]
    if best_score >= 0.9:
        matches_tfidf.append((df1.iloc[i]["CMTE_ID"], df2.iloc[best_match_idx]["RIC"]))
    else:
        unmatched.append((df1.iloc[i]["CONNECTED_ORG_NM"]))

df_tfidf_matches = pd.DataFrame(matches_tfidf, columns=["CMTE_ID", "RIC"])
df_unmatched = pd.DataFrame(unmatched, columns = ["connected_org"])
df_tfidf_matches = df_tfidf_matches.drop_duplicates()
df_unmatched = df_unmatched.drop_duplicates()

3198it [00:00, 12244.50it/s]


TF-IDF Cosine Similarity Matches:
        CMTE_ID     RIC
0    C00000042   ITW.N
1    C00001289   CVX.N
2    C00002790   OLN.N
3    C00004960   COP.N
4    C00007948    WY.N
..         ...     ...
491  C00888842    PR.N
492  C00889568  CPNG.N
493  C00893578   PRM.N
494  C00893925  CIVI.N
495  C00894345  BANC.N

[496 rows x 2 columns]


In [7]:
cpac = pd.merge(df1, df_tfidf_matches, how="left", on="CMTE_ID")
cpac.to_csv("../data/cpac/corporate_pacs.csv", index=False)

In [85]:
def clean_firm(text):
    text = text.lower().strip()
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'\.', '', text)
    text = re.sub(r'\b(federal|political|committee|action|election|politi|politic)\b', '', text)
    return text.strip()

In [ ]:
df_unmatched["company_name"] = df_unmatched["connected_org"].apply(lambda x: preprocess(x, business=False, symbol=False, period=True))
df_unmatched.dropna(subset="company_name")

In [96]:
df_unmatched.to_csv("corporate_pacs_unmatched.csv", index=False)

### Search Unmatched Firms

In [133]:
import lseg.data as ld
from lseg.data.content import search
ld.open_session()

<lseg.data.session.Definition object at 0x241badfd4c0 {name='workspace'}>

In [90]:
df_unmatched["matched_company"] = None
df_unmatched["PI"] = None

In [ ]:
for index, row in tqdm(df_unmatched.iterrows()):
    if row["matched_company"] is not None:
        continue
    response = search.Definition(
        f'"{row["company_name"]}"', view = search.Views.ORGANISATIONS
    ).get_data()
    df_response = response.data.df
    if len(df_response) == 0:
        response = search.Definition(
            row["company_name"], view = search.Views.ORGANISATIONS
        ).get_data()
        df_response = response.data.df
    if len(df_response) > 0:
        df_unmatched.at[index, "matched_company"] = df_response.loc[0, "DocumentTitle"]
        df_unmatched.at[index, "PI"] = df_response.loc[0, "PI"]
    else:
        df_unmatched.at[index, "matched_company"] = ""
        df_unmatched.at[index, "PI"] = ""
    df_unmatched.to_csv("corporate_pacs_unmatched.csv", index=False)

2574it [17:08,  2.50it/s]


#### Check Matched Names

In [325]:
from Levenshtein import ratio

In [430]:
df_unmatched = pd.read_csv("../data/cpac/corporate_pacs_unmatched.csv", dtype={"PI": str})
df_unmatched["matched_company"] = df_unmatched["matched_company"].fillna("")

In [431]:
def clean_company_name(text):
    if pd.isna(text):
        return "", False
    if not ((", Private Company" in text) or (", Public Company" in text)):
        return "", False
    text = text.lower()
    text = re.sub(r'\([^)]*\)', ' ', text)
    if "private company" in text:
        return text.replace(", private company", "").replace(", delisted", ""), False
    return text.replace(", public company", ""), True

def similarity_ratios(text1, text2):
    if ((len(text1) == 0) | (len(text2) == 0)):
        return 0, 0
    text1 = preprocess(text1)
    text2 = preprocess(text2)
    levenshtein = ratio(text1, text2)
    set1, set2 = set(text1.split()), set(text2.split())
    jaccard = len(set1 & set2) / len(set1 | set2)
    return levenshtein, jaccard

In [432]:
df_unmatched["cleaned_company"], df_unmatched["public_company"] = zip(*df_unmatched["matched_company"].apply(clean_company_name))

In [433]:
df_unmatched["levenshein"], df_unmatched["jaccard"] = zip(
    *df_unmatched.apply(lambda x: similarity_ratios(x["company_name"], x["cleaned_company"]), axis=1))

In [434]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(
    df_unmatched["company_name"].apply(preprocess).tolist() + 
    df_unmatched["cleaned_company"].apply(preprocess).tolist())
cosine_scores = cosine_similarity(tfidf_matrix[:len(df_unmatched)], tfidf_matrix[len(df_unmatched):])

df_unmatched["cosine"] = [cosine_scores[i, i] for i in range(len(df_unmatched))]

In [435]:
lseg_matched = df_unmatched.loc[(df_unmatched["levenshein"] >= 0.9) | (df_unmatched["cosine"] >= 0.9), ]
lseg_unmatched = df_unmatched.loc[(df_unmatched["levenshein"] < 0.9) & (df_unmatched["cosine"] < 0.9), ]
lseg_matched.to_csv("../data/cpac/lseg_matched.csv", index=False)
lseg_unmatched.to_csv("../data/cpac/lseg_unmatched.csv", index=False)

### Collect Public Company Information

In [408]:
lseg_matched_public = lseg_matched.loc[lseg_matched["public_company"],["connected_org", "cleaned_company", "PI"]].copy()
lseg_matched_public["matched_equity"] = None
lseg_matched_public["PermID"] = None
lseg_matched_public["RIC"] = None

In [410]:
for index, row in tqdm(lseg_matched_public.iterrows()):
    response = search.Definition(
        f'"{row["cleaned_company"]}"', view = search.Views.EQUITY_QUOTES,
        filter="SearchAllCategory eq 'Equities'"
    ).get_data()
    df_response = response.data.df
    if len(df_response) > 0:
        if row["cleaned_company"] not in df_response.loc[0, "DocumentTitle"].lower():
            response = search.Definition(
                f'"{row["cleaned_company"]}" AND "Share"', view = search.Views.EQUITY_QUOTES
                ).get_data()
            df_response = response.data.df
        lseg_matched_public.at[index, "matched_equity"] = df_response.loc[0, "DocumentTitle"]
        lseg_matched_public.at[index, "PI"] = df_response.loc[0, "PI"]
        lseg_matched_public.at[index, "PermID"] = df_response.loc[0, "PermID"]
        lseg_matched_public.at[index, "RIC"] = df_response.loc[0, "RIC"]
    lseg_matched_public.to_csv("../data/cpac/lseg_matched_public.csv", index=False)

199it [01:55,  1.72it/s]


In [411]:
cpac = pd.read_csv("../data/cpac/corporate_pacs.csv")

In [412]:
cpac = cpac.merge(lseg_matched_public[["connected_org", "RIC"]], 
                  left_on="CONNECTED_ORG_NM", right_on="connected_org", how="outer",
                  suffixes=('', '_new'))
cpac['RIC'] = cpac['RIC'].fillna(cpac['RIC_new'])
cpac.drop(columns=["connected_org", "RIC_new"], inplace=True)
cpac = cpac.sort_values("CMTE_ID").reset_index(drop=True)

In [413]:
cpac.to_csv("../data/cpac/corporate_pacs.csv", index=False)

### Group Private Companies by PermID

In [436]:
lseg_matched_private = lseg_matched.loc[~lseg_matched["public_company"],["connected_org", "cleaned_company", "PI"]].copy()

In [437]:
lseg_matched_private["PermID"] = lseg_matched_private["PI"].apply(lambda x: x if len(x) == 10 else None)
lseg_matched_private.to_csv("../data/cpac/lseg_matched_private.csv", index=False)

In [457]:
cpac = pd.read_csv("../data/cpac/corporate_pacs.csv")
cpac = cpac.merge(lseg_matched_private[["connected_org", "PermID"]], 
                  left_on="CONNECTED_ORG_NM", right_on="connected_org", how="outer",
                  suffixes=('', '_new'))
cpac.drop(columns=["connected_org"], inplace=True)
cpac = cpac.sort_values("CMTE_ID").reset_index(drop=True)
cpac.to_csv("../data/cpac/corporate_pacs.csv", index = False)

In [458]:
lseg_matched_private_handcode = lseg_matched_private.loc[lseg_matched_private["PermID"].isna(), ]
lseg_matched_private_handcode.to_csv("../data/cpac/lseg_matched_private_handcode.csv", index=False)

### Try Unmatched Firms Again

In [396]:
lseg2 = pd.read_csv("../data/cpac/lseg_unmatched.csv", dtype={"PI": str})
lseg2["company_name"] = lseg2["connected_org"].apply(preprocess)

In [398]:
for index, row in tqdm(lseg2.iterrows()):
    response = search.Definition(
        f'"{row["company_name"]}"', view = search.Views.ORGANISATIONS
    ).get_data()
    df_response = response.data.df
    if len(df_response) > 0:
        lseg2.at[index, "matched_company"] = df_response.loc[0, "DocumentTitle"]
        lseg2.at[index, "PI"] = df_response.loc[0, "PI"]
    else:
        lseg2.at[index, "matched_company"] = ""
        lseg2.at[index, "PI"] = ""

956it [09:06,  1.75it/s]


In [399]:
lseg2["cleaned_company"], lseg2["public_company"] = zip(*lseg2["matched_company"].apply(clean_company_name))
lseg2["levenshein"], lseg2["jaccard"] = zip(
    *lseg2.apply(lambda x: similarity_ratios(x["company_name"], preprocess(x["cleaned_company"])), axis=1))

In [400]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(
    lseg2["company_name"].apply(preprocess).tolist() + 
    lseg2["cleaned_company"].apply(preprocess).tolist())
cosine_scores = cosine_similarity(tfidf_matrix[:len(lseg2)], tfidf_matrix[len(lseg2):])

lseg2["cosine"] = [cosine_scores[i, i] for i in range(len(lseg2))]

lseg2.to_csv("../data/cpac/lseg2.csv", index=False)

In [401]:
lseg2_matched = lseg2.loc[(lseg2["levenshein"] >= 0.9) | (lseg2["cosine"] >= 0.9), ]
lseg2_unmatched = lseg2.loc[(lseg2["levenshein"] < 0.9) & (lseg2["cosine"] < 0.9), ]
lseg2_matched.to_csv("../data/cpac/lseg2_matched.csv", index=False)
lseg2_unmatched.to_csv("../data/cpac/lseg2_unmatched.csv", index=False)

In [402]:
lseg2_matched_public = lseg2_matched.loc[lseg2_matched["public_company"],["connected_org", "cleaned_company", "PI"]].copy()
lseg2_matched_private = lseg2_matched.loc[~lseg2_matched["public_company"],["connected_org", "cleaned_company", "PI"]].copy()
lseg2_matched_public["matched_equity"] = None
lseg2_matched_public["PermID"] = None
lseg2_matched_public["RIC"] = None

In [403]:
for index, row in tqdm(lseg2_matched_public.iterrows()):
    response = search.Definition(
        f'"{row["cleaned_company"]}"', view = search.Views.EQUITY_QUOTES,
        filter="SearchAllCategory eq 'Equities'"
    ).get_data()
    df_response = response.data.df
    if len(df_response) > 0:
        if row["cleaned_company"] not in df_response.loc[0, "DocumentTitle"].lower():
            response = search.Definition(
                f'"{row["cleaned_company"]}" AND "Share"', view = search.Views.EQUITY_QUOTES
                ).get_data()
            df_response = response.data.df
        lseg2_matched_public.at[index, "matched_equity"] = df_response.loc[0, "DocumentTitle"]
        lseg2_matched_public.at[index, "PI"] = df_response.loc[0, "PI"]
        lseg2_matched_public.at[index, "PermID"] = df_response.loc[0, "PermID"]
        lseg2_matched_public.at[index, "RIC"] = df_response.loc[0, "RIC"]
    lseg2_matched_public.to_csv("../data/cpac/lseg2_matched_public.csv", index=False)

14it [00:12,  1.11it/s]


In [512]:
lseg2_matched_private["PermID"] = lseg2_matched_private["PI"].apply(lambda x: x if len(x) == 10 else None)
lseg2_matched_private.to_csv("../data/cpac/lseg2_matched_private.csv", index=False)

In [405]:
cpac = pd.read_csv("../data/cpac/corporate_pacs.csv", dtype={"PermID": str})
cpac = cpac.merge(lseg2_matched_public[["connected_org", "RIC"]], 
                  left_on="CONNECTED_ORG_NM", right_on="connected_org", how="outer",
                  suffixes=('', '_new'))
cpac['RIC'] = cpac['RIC'].fillna(cpac['RIC_new'])
cpac.drop(columns=["connected_org", "RIC_new"], inplace=True)
cpac = cpac.sort_values("CMTE_ID").reset_index(drop=True)

In [459]:
cpac = cpac.merge(lseg2_matched_private[["connected_org", "PermID"]], 
                  left_on="CONNECTED_ORG_NM", right_on="connected_org", how="outer",
                  suffixes=('', '_new'))
cpac['PermID'] = cpac['PermID'].fillna(cpac['PermID_new'])
cpac.drop(columns=["connected_org", "PermID_new"], inplace=True)
cpac = cpac.sort_values("CMTE_ID").reset_index(drop=True)
cpac.to_csv("../data/cpac/corporate_pacs.csv", index=False)

In [416]:
lseg2_matched_private_handcode = lseg2_matched_private.loc[lseg2_matched_private["PermID"].isna(), ]
lseg2_matched_private_handcode.to_csv("../data/cpac/lseg2_matched_private_handcode.csv", index=False)

### Third Try at Remaining Unmatched

In [498]:
lseg3 = pd.read_csv("../data/cpac/lseg2_unmatched.csv", dtype={"PI": str})
lseg3["company_name"] = lseg3["connected_org"].apply(lambda x: preprocess(x, business=False))

In [499]:
for index, row in tqdm(lseg3.iterrows()):
    response = search.Definition(
        f'"{row["company_name"]}"', view = search.Views.ORGANISATIONS
    ).get_data()
    df_response = response.data.df
    if len(df_response) > 0:
        lseg3.at[index, "matched_company"] = df_response.loc[0, "DocumentTitle"]
        lseg3.at[index, "PI"] = df_response.loc[0, "PI"]
    else:
        lseg3.at[index, "matched_company"] = ""
        lseg3.at[index, "PI"] = ""

806it [07:44,  1.74it/s]


In [501]:
lseg3["cleaned_company"], lseg3["public_company"] = zip(*lseg3["matched_company"].apply(clean_company_name))
lseg3["levenshein"], lseg3["jaccard"] = zip(
    *lseg3.apply(lambda x: similarity_ratios(x["company_name"], preprocess(x["cleaned_company"])), axis=1))

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(
    lseg3["company_name"].apply(preprocess).tolist() + 
    lseg3["cleaned_company"].apply(preprocess).tolist())
cosine_scores = cosine_similarity(tfidf_matrix[:len(lseg3)], tfidf_matrix[len(lseg3):])

lseg3["cosine"] = [cosine_scores[i, i] for i in range(len(lseg3))]

In [506]:
lseg3_matched = lseg3.loc[(lseg3["levenshein"] >= 0.9) | (lseg3["cosine"] >= 0.9), ]
lseg3_unmatched = lseg3.loc[(lseg3["levenshein"] < 0.9) & (lseg3["cosine"] < 0.9), ]
lseg3_matched.to_csv("../data/cpac/lseg3_matched.csv", index=False)
lseg3_unmatched.to_csv("../data/cpac/lseg3_unmatched.csv", index=False)

In [514]:
lseg3_matched_private = lseg3_matched[["connected_org", "cleaned_company", "PI"]].copy()
lseg3_matched_private["PermID"] = lseg3_matched_private["PI"].apply(lambda x: x if len(x) == 10 else None)
lseg3_matched_private.to_csv("../data/cpac/lseg3_matched_private.csv", index=False)

In [515]:
lseg3_matched_private_handcode = lseg3_matched_private.loc[lseg3_matched_private["PermID"].isna(), ]
lseg3_matched_private_handcode.to_csv("../data/cpac/lseg3_matched_private_handcode.csv", index=False)

In [523]:
cpac = pd.read_csv("../data/cpac/corporate_pacs.csv", dtype={"PermID": str})
cpac = cpac.merge(lseg3_matched_private[["connected_org", "PermID"]], 
                  left_on="CONNECTED_ORG_NM", right_on="connected_org", how="outer",
                  suffixes=('', '_new'))
cpac['PermID'] = cpac['PermID'].fillna(cpac['PermID_new'])
cpac.drop(columns=["connected_org", "PermID_new"], inplace=True)
cpac = cpac.sort_values("CMTE_ID").reset_index(drop=True)
cpac.to_csv("../data/cpac/corporate_pacs.csv", index=False)

### Save All Firms that Need Handcoding

In [533]:
handcode = pd.concat([lseg_matched_private_handcode,
                      lseg2_matched_private_handcode,
                      lseg3_matched_private_handcode,
                      lseg3_unmatched[["connected_org"]]])
handcode.drop(columns="PermID", inplace=True)

In [ ]:
cpac = pd.read_csv("../data/cpac/corporate_pacs.csv")
handcode = handcode.merge(cpac[["CONNECTED_ORG_NM", "CMTE_NM"]], how="left",
                          left_on = "connected_org", right_on = "CONNECTED_ORG_NM")
handcode.drop(columns="CONNECTED_ORG_NM", inplace=True)

In [15]:
handcode.to_excel("../data/cpac/handcode.xlsx", index=False)

### Merge in Handcoded Data

In [24]:
cpac = pd.read_csv("../data/cpac/corporate_pacs.csv", dtype={"PermID":str})
handcode = pd.read_excel("../data/cpac/handcode.xlsx", dtype={"PermID": str})

In [25]:
cpac = pd.merge(cpac, handcode, how = "left",
                left_on = ["CONNECTED_ORG_NM", "CMTE_NM"],
                right_on = ["connected_org", "CMTE_NM"],
                suffixes = ('', '_right'))
cpac.drop(columns = "connected_org", inplace = True)

In [26]:
cpac["PermID"] = cpac["PermID"].fillna(cpac["PermID_right"])
cpac["RIC"] = cpac["RIC"].fillna(cpac["RIC_right"])
cpac.drop(columns = ["PermID_right", "RIC_right", "cleaned_company", "PI"], inplace = True)

In [ ]:
unmatched = cpac[cpac[["RIC", "PermID", "Parent", "Website"]].isna().all(axis=1)]
no_connect_org = unmatched.loc[unmatched["company_name"] == "none", ]
no_connect_org.to_excel("../data/cpac/no_connected_org.xlsx", index=False)

In [58]:
no_connect_org = pd.read_excel("../data/cpac/no_connected_org.xlsx", dtype={"PermID": str})

In [69]:
cpac = pd.merge(cpac, no_connect_org[["CMTE_ID", "RIC", "PermID", "Parent", "Website"]], how = "left",
                on="CMTE_ID",
                suffixes = ('', '_right'))

In [71]:
cpac["PermID"] = cpac["PermID"].fillna(cpac["PermID_right"])
cpac["RIC"] = cpac["RIC"].fillna(cpac["RIC_right"])
cpac["Parent"] = cpac["Parent"].fillna(cpac["Parent_right"])
cpac["Website"] = cpac["Website"].fillna(cpac["Website_right"])
cpac.drop(columns = ["PermID_right", "RIC_right", "Parent_right", "Website_right"], inplace=True)

In [75]:
cpac = cpac.dropna(subset=["RIC", "PermID", "Parent", "Website"], how = "all")
cpac.to_csv("../data/cpac/corporate_pacs.csv", index=False)

### Expand Matched PACs to Full Dataset

In [109]:
cpac = pd.read_csv("../data/cpac/corporate_pacs.csv", dtype={"PermID":str})
cpac_full = pd.read_csv("../data/cpac/corporate_pacs_full.csv")

In [ ]:
cpac["cmte_name"] = cpac["CMTE_NM"].apply(preprocess)
cpac["org_name"] = cpac["CONNECTED_ORG_NM"].apply(preprocess)
cpac_full["cmte_name"] = cpac_full["CMTE_NM"].apply(preprocess)
cpac_full["org_name"] = cpac_full["CONNECTED_ORG_NM"].apply(preprocess)

In [139]:
cpac_merged = pd.merge(cpac_full, 
                       cpac[["CMTE_ID", "cmte_name", "org_name", "company_name", "RIC", "PermID", "Parent", "Website"]],
                       how = "left",
                       on = ["CMTE_ID", "cmte_name", "org_name"]).drop(columns=["cmte_name", "org_name"])

In [145]:
cpac_merged.to_csv("../data/cpac/corporate_pacs_merged.csv", index=False)

In [147]:
cmte_id = cpac_merged.loc[cpac_merged["company_name"].isna(),"CMTE_ID"].unique().tolist()
cpac_full_manual = cpac_merged.loc[cpac_merged["CMTE_ID"].isin(cmte_id),]
cpac_full_manual.to_excel("../data/cpac/corporate_pacs_full_manual.xlsx", index=False)

### Merge Handcoded Full Dataset back

In [148]:
cpac_merged = pd.read_csv("../data/cpac/corporate_pacs_merged.csv", dtype={"PermID": str})
cpac_full_manual = pd.read_excel("../data/cpac/corporate_pacs_full_manual.xlsx", dtype={"PermID": str})

In [151]:
cpac_merged = pd.merge(cpac_merged, cpac_full_manual,
                       how = "left",
                       on = ["CMTE_ID", "CMTE_NM", "CONNECTED_ORG_NM", "CORPORATION", "YEAR"],
                       suffixes = ("", "_right"))

In [ ]:
cpac_merged["RIC"] = cpac_merged["RIC"].fillna(cpac_merged["RIC_right"])
cpac_merged["PermID"] = cpac_merged["PermID"].fillna(cpac_merged["PermID_right"])
cpac_merged["Parent"] = cpac_merged["Parent"].fillna(cpac_merged["Parent_right"])
cpac_merged["Website"] = cpac_merged["Website"].fillna(cpac_merged["Website_right"])
cpac_merged.drop(columns=["company_name", "company_name_right", "RIC_right", "PermID_right", "Parent_right", "Website_right"], inplace=True)
cpac_merged = cpac_merged.dropna(subset=["RIC", "PermID", "Parent", "Website"], how="all").reset_index(drop=True)

In [162]:
cpac_merged.to_csv("../data/cpac/corporate_pacs_2004_2024.csv", index=False)